# Leaderboard Pipeline 06 — Controlled Experiment Matrix

This notebook defines the minimum set of training runs needed before spending the final two submissions.

No experiment is submitted directly. First compare OOF Macro-F1, per-class F1, and fold stability.


In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "leaderboard_pipeline":
    REPO_ROOT = REPO_ROOT.parents[1]
elif REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

DATA_ROOT = REPO_ROOT / "BDC2026"
CLEAN_ROOT = REPO_ROOT / "BDC2026_clean_v1"
PIPELINE_ROOT = REPO_ROOT / "leaderboard_pipeline_outputs"
PIPELINE_ROOT.mkdir(parents=True, exist_ok=True)
LABEL2ID = {"0_Recyclable": 0, "1_Electronic": 1, "2_Organic": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("CLEAN_ROOT:", CLEAN_ROOT)


In [ ]:
import pandas as pd

STAGE_DIR = PIPELINE_ROOT / "06_experiments"
STAGE_DIR.mkdir(parents=True, exist_ok=True)
FOLD_CSV = PIPELINE_ROOT / "05_folds" / "train_folds_duplicate_aware.csv"
assert CLEAN_ROOT.exists(), f"Create clean dataset first: {CLEAN_ROOT}"
assert FOLD_CSV.exists(), f"Run notebook 05 first: {FOLD_CSV}"

experiments = pd.DataFrame([
    {"id":"E00", "status":"completed", "dataset":"original", "resolution":384, "seed":42, "imbalance":"class_weights", "purpose":"public baseline 98.42", "priority":0},
    {"id":"E01", "status":"planned", "dataset":"clean_v1", "resolution":384, "seed":42, "imbalance":"class_weights", "purpose":"cleaning + duplicate-aware folds; S02 candidate", "priority":1},
    {"id":"E02", "status":"planned", "dataset":"clean_v1", "resolution":384, "seed":123, "imbalance":"class_weights", "purpose":"seed diversity for final ensemble", "priority":2},
    {"id":"E03", "status":"optional", "dataset":"clean_v1", "resolution":384, "seed":777, "imbalance":"weighted_sampler", "purpose":"imbalance-method diversity", "priority":3},
    {"id":"E04", "status":"optional", "dataset":"clean_v1", "resolution":224, "seed":42, "imbalance":"class_weights", "purpose":"resolution diversity", "priority":4},
])
experiments.to_csv(STAGE_DIR / "experiment_registry.csv", index=False)
experiments


In [ ]:
def train_command(row):
    output = f"./experiments/{row['id']}_{row['dataset']}_{row['resolution']}_seed{row['seed']}"
    imbalance = (
        "--use-class-weights"
        if row["imbalance"] == "class_weights"
        else "--use-weighted-sampler --sampler-weight-mode sqrt_inverse"
    )
    batch, valid_batch, accumulation = (
        (2, 4, 8) if row["resolution"] >= 384 else (4, 8, 4)
    )
    inner = (
        f"python scripts/train_with_precomputed_folds.py --data-root {CLEAN_ROOT} "
        f"--fold-csv {FOLD_CSV} --output-dir {output} "
        f"--image-size {row['resolution']} --epochs 25 "
        f"--batch-size {batch} --valid-batch-size {valid_batch} "
        f"--grad-accum {accumulation} --seed {row['seed']} {imbalance} "
        "--scheduler plateau --early-stopping-patience 8"
    )
    return (
        "python scripts/launch_train_auto_gpus.py --max-gpus 3 --min-gpus 1 "
        "--min-free-mb 30000 --max-utilization 30 -- " + inner
    )

planned = experiments[experiments["status"].isin(["planned", "optional"])].copy()
planned["command"] = planned.apply(train_command, axis=1)
display(planned[["id", "priority", "purpose", "command"]])


In [ ]:
script_lines = ["#!/usr/bin/env bash", "set -euo pipefail", ""]
for _, row in planned.sort_values("priority").iterrows():
    script_lines += [f"# {row['id']}: {row['purpose']}", row["command"], ""]

command_path = STAGE_DIR / "generated_commands.sh"
command_path.write_text("\n".join(script_lines), encoding="utf-8")
print("Saved", command_path)
print("Run E01 first. Do not run optional experiments until E01 OOF analysis is complete.")


In [ ]:
decision_checklist = pd.DataFrame([
    {"candidate":"E01", "requirement":"OOF Macro-F1 gain", "threshold":">= 0.0010", "passed":""},
    {"candidate":"E01", "requirement":"folds improved", "threshold":">= 3 of 5", "passed":""},
    {"candidate":"E01", "requirement":"weakest class F1 drop", "threshold":"<= 0.0005", "passed":""},
    {"candidate":"E01", "requirement":"cleaning audit reviewed", "threshold":"yes", "passed":""},
    {"candidate":"S02", "requirement":"submission schema and order validated", "threshold":"yes", "passed":""},
])
decision_checklist.to_csv(STAGE_DIR / "S02_decision_checklist.csv", index=False)
decision_checklist


## Compute-efficient order

1. Train **E01 only**.
2. Compare E01 with the baseline using OOF and fold metrics.
3. If E01 is clearly better, train E02 for ensemble diversity.
4. Run E03 or E04 only when disagreement analysis shows that more diversity is useful.
5. Do not use a leaderboard submission to choose among E01–E04.

Submission S02 should be the best validated single model. Submission S03 should be the final OOF-optimized ensemble.
